# 6. Combining experiments, and how firm a result is

Two things that matter once a search produces numbers someone might act on: **where two
experiments can share ground**, and **how much the answer depends on assumptions**.

The second is the more important, and it is the one most easily skipped.

In [1]:
import numpy as np

from oroscope import combine_experiments as combine, physics

## Combining is an overlay, and alignment is not optional

Each experiment is one run of the searcher with its own configuration, so combining them
is an overlay of the masks those runs produce. Three questions get different answers:

- **joint** — terrain satisfying *every* experiment. One site, one road, one power feed,
  two experiments.
- **union** — terrain satisfying *any*. How much of the region is useful to the
  programme as a whole.
- **each alone** — and what each would lose by being confined to the joint area.

The inputs must be pixel-aligned: same shape, same pixel size, same corner. That is
checked and **refused** rather than resampled, because two runs on differently-cropped
DEMs would silently compare the wrong ground.

In [2]:
# What the check actually compares: the six affine terms of the world file
world_a = (1/3600, 0.0, 0.0, -1/3600, -72.4, -15.3)
world_b = (1/3600, 0.0, 0.0, -1/3600, -72.1, -15.3)      # shifted 0.3 deg east

runs = [{"dir": "run_a", "mask": np.zeros((10, 10), bool), "world": world_a},
        {"dir": "run_b", "mask": np.zeros((10, 10), bool), "world": world_b}]

try:
    combine.check_alignment(runs)
except SystemExit as exc:
    print("refused, correctly:\n")
    print(exc)

refused, correctly:

masks do not cover the same ground: upper-left x is -72.4 in run_a and -72.1 in run_b.


Same shape, same pixel size, different ground — and it says so rather than overlaying
them.

## Reading a co-location result

On the Colca crop, with both experiments run over identical terrain:

| | area | sites | capacity | of its own area in the joint |
|---|---|---|---|---|
| GRAND | 4569.4 km² | 1 | 5315 | 2.7% |
| TAMBO | 203.0 km² | 16 | 10 437 | 60.7% |
| **joint** | 123.3 km² | | | Jaccard 0.027 |

The interesting part is *why* the joint is small. Three fifths of TAMBO-viable ground is
also GRAND-viable, but the two deployable **slope bands barely overlap** — GRAND's 3–25°
against Colca's ~40° walls leaves only a 20–25° sliver. Co-location is decided by slope,
not by arrival geometry.

> An earlier version of this table reported TAMBO at 44.5 km² and the joint at 26.4.
> Both were wrong. `load_run` took the alphabetically first `.tif` in a run directory,
> and a directory re-run since the project was renamed holds both
> `oroscope_results_*.tif` and a stale `grand_search_results_*.tif` — the legacy prefix
> sorts first, so the overlay quietly used a superseded mask. Nothing failed; the
> report simply described a run that no longer existed. It is worth knowing that this
> class of fault produces a plausible number rather than an error.

In [3]:
grand = (3.0, 25.0)
tambo = (20.0, 60.0)
lo, hi = max(grand[0], tambo[0]), min(grand[1], tambo[1])
print(f"GRAND deployable band: {grand[0]:.0f}-{grand[1]:.0f} deg")
print(f"TAMBO wall band:       {tambo[0]:.0f}-{tambo[1]:.0f} deg")
print(f"overlap:               {lo:.0f}-{hi:.0f} deg  ({hi-lo:.0f} deg wide)")

GRAND deployable band: 3-25 deg
TAMBO wall band:       20-60 deg
overlap:               20-25 deg  (5 deg wide)


## How firm is a result?

`oroscope-sensitivity` varies one parameter at a time about a baseline and tabulates how
much each moves the answer. Run against a real TAMBO baseline, the verdict was blunt:

| parameter | | | | |
|---|---|---|---|---|
| `min_score` | 0.0 → **65 268** | 0.2 → **25 635** | 0.35 → **10 437** | 0.5 → **0** |
| `min_target_slope_deg` | 0° → **18 622** | 15° → **14 720** | 25° → **10 437** | 35° → **2814** |
| `decay_spectral_index` | 1.5 → **6853** | 2.0 → **10 437** | 2.7 → **11 349** | |

Both cuts sit near a cliff: `min_score` runs 6.25× to zero across the swept range and
`min_target_slope_deg` 1.78× to 0.27×, about a baseline of 10 437.

**A fourth row used to head this table, and its disappearance is the point.**
`decay_energy_pev` once ran 3 PeV → 10 878 detector positions and 100 PeV → zero: a
single representative energy was choosing the answer rather than approximating it. The
decay term is folded over a power-law spectrum now, so the same question is asked as
`decay_spectral_index` above and the answer varies by **1.66×** across a plausible range
of index rather than without bound.

> Sweeping `decay_energy_pev` today reports **1.00× at every value**, and that is not
> robustness — it is the parameter being ignored. `scoring.score_candidates` prefers
> `decay_energy_min_pev`/`max_pev` when both are set, which this configuration sets, so
> the single-energy branch is never reached. A sweep over a parameter the configuration
> does not consult looks exactly like a sweep over one that does not matter.

In [4]:
crossing_m = 3000.0
print(f"P(tau decays within a {crossing_m/1000:.0f} km crossing):\n")
for e in (3.0, 10.0, 55.0, 100.0, 1000.0):
    L = physics.tau_decay_length_m(e)
    p = 1 - np.exp(-crossing_m / L)
    bar = "#" * int(round(p * 40))
    print(f"{e:>7.0f} PeV  {p:5.3f}  {bar}")

P(tau decays within a 3 km crossing):

      3 PeV  1.000  ########################################
     10 PeV  0.998  ########################################
     55 PeV  0.672  ###########################
    100 PeV  0.458  ##################
   1000 PeV  0.059  ##


That is a factor of seventeen inside one experiment's energy reach — and it is invisible
to every other term in the score.

**So: fold over the real spectrum before quoting a capacity.** A number computed at one
representative energy is a property of the energy chosen, not of the terrain.

## What to distrust in your own results

Three things this project measured about itself, worth checking in any search:

1. **Reported area is not physics-accepted area.** Morphological closing more than
   doubles it — measured at 2.35× with a stride-1 control run. Closing is not wrong; a
   site has to be a deployable region rather than a scatter of pixels. But the two
   numbers are different and should not be conflated.
2. **Candidate striding is unbiased** — acceptance is identical at strides 1 and 5, and
   the stride-corrected area matches the stride-1 truth to 0.05%. So that one *is*
   safe, *with the caveat below*.
3. **The closing element and the stride interact.** That striding result was measured
   with GRAND's 1 km closing element. Each run's own funnel gives the factor directly,
   and the two Colca configs still disagree, though far less than they used to: GRAND's
   mask is 2.25× its stride-corrected accepted set — an independent check on the 2.35×
   above — while TAMBO's is **0.97×**, because a 150 m element is five pixels and only
   just bridges the gaps stride 5 leaves. TAMBO's area is therefore still a *lower*
   bound, but by 1.51× rather than the 4.75× a 100 m element cost.
4. **Area and capacity are measured on different grids** at `downsample_factor > 1`, so
   a feature a few pixels wide loses area it keeps detectors on.
5. **Not every site in the results file is in the result.** `sites` lists everything
   that cleared the thresholds; with `stop_at_target`, only the first *n* were
   selected. Filter on each record's `selected` flag before totalling anything.

Every run reports 3 for itself, in its own summary. `docs/assumptions.rst` is the full
list, and it is deliberately blunt.

## Where to go next

- The **[assumptions and limitations](https://mbustama.github.io/oroscope/assumptions.html)**
  page — what the numbers rest on.
- The **[physics](https://mbustama.github.io/oroscope/physics.html)** page — the
  derivation behind every criterion.

## Where to go next

- **[7. Animating the mechanism](07_animating_the_mechanism.ipynb)** — the parts of
  all this that a still picture explains badly, as eight short films.
- **[8. Explaining a run](08_explaining_a_run.ipynb)** — driving the pipeline from
  Python, and reading what it says about a run that succeeds and one that does not.

---

*Part of the [Oroscope](https://github.com/mbustama/oroscope) tutorials. Previous: [GRAND and TAMBO](05_grand_and_tambo.ipynb). Next: [Animating the mechanism](07_animating_the_mechanism.ipynb). Full API reference: [oroscope docs](https://mbustama.github.io/oroscope/functions.html).*